# Data Preparations for Time Series Data

This code is meant to prepare the data for the dependent variable and create a simple time series data table for the preliminary aspects of the project.

In [5]:
import pandas as pd

# Load Excel file
file_path = "ppms idealista.xlsx"

# Read all sheets into dictionary
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

# Store each sheet in list
dfs = []

for sheet in sheet_names:
    df_temp = pd.read_excel(xls, sheet_name=sheet)
    df_temp["Year"] = sheet  # Keep year info from sheet name
    dfs.append(df_temp)

# Combine into single dataframe
df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df.shape)
print(df.head())


Combined shape: (2128, 12)
            Location  Price 2014  Year  Price 2015  Price 2016  Price 2017  \
0              Italy         8.6  2014         NaN         NaN         NaN   
1            Abruzzo         6.4  2014         NaN         NaN         NaN   
2    Chieti province         5.9  2014         NaN         NaN         NaN   
3             Chieti         6.2  2014         NaN         NaN         NaN   
4  L'Aquila province         6.9  2014         NaN         NaN         NaN   

   Price 2018  Price 2019  Price 2020  Price 2021  Price 2022  Price 2023  
0         NaN         NaN         NaN         NaN         NaN         NaN  
1         NaN         NaN         NaN         NaN         NaN         NaN  
2         NaN         NaN         NaN         NaN         NaN         NaN  
3         NaN         NaN         NaN         NaN         NaN         NaN  
4         NaN         NaN         NaN         NaN         NaN         NaN  


In [6]:
df["Location"] = (
    df["Location"]
    .astype(str)
    .str.strip()
)
print(df["Location"].unique())

<StringArray>
[             'Italy',            'Abruzzo',    'Chieti province',
             'Chieti',  'L'Aquila province',           'L'Aquila',
   'Pescara province',            'Pescara',    'Teramo province',
             'Teramo',
 ...
             'Matera',            'Isernia',      'Bolzano-Bozen',
 'Reggio nell'Emilia',          'La Spezia',            'Cremona',
             'Biella',          'Agrigento',            'Trapani',
              'Massa']
Length: 229, dtype: str


In [7]:
df_provinces = df[
    (df["Location"].str.contains("province", case=False, na=False)) |
    (df["Location"].str.contains("forlì-cesena", case=False, na=False))|
    (df["Location"].str.contains("barletta", case=False, na=False)) |
    (df["Location"].str.contains("reggio emilia", case=False, na=False)) |    
    (df["Location"].str.contains("brianza", case=False, na=False)) |
    (df["Location"].str.contains("urbino", case=False, na=False)) |
    (df["Location"].str.contains("ossola", case=False, na=False)) |
    (df["Location"].str.contains("massa-carrara", case=False, na=False))
    ].copy()


In [8]:
df_provinces["Province"] = (
    df_provinces["Location"]
    .str.replace(" province", "", case=False, regex=False)
    .str.strip()
)


In [9]:
print("Unique provinces:", df_provinces["Province"].nunique())
print(sorted(df_provinces["Province"].unique()))

Unique provinces: 106
['Agrigento', 'Alessandria', 'Ancona', 'Aosta', 'Arezzo', 'Ascoli Piceno', 'Asti', 'Avellino', 'Bari', 'Barletta-Andria-Trani', 'Belluno', 'Benevento', 'Bergamo', 'Biella', 'Bologna', 'Bolzano-Bozen', 'Brescia', 'Brindisi', 'Cagliari', 'Caltanissetta', 'Campobasso', 'Caserta', 'Catania', 'Catanzaro', 'Chieti', 'Como', 'Cosenza', 'Cremona', 'Crotone', 'Cuneo', 'Enna', 'Fermo', 'Ferrara', 'Firenze', 'Foggia', 'Forlì-Cesena', 'Frosinone', 'Genova', 'Gorizia', 'Grosseto', 'Imperia', 'Isernia', "L'Aquila", 'La Spezia', 'Latina', 'Lecce', 'Lecco', 'Livorno', 'Lodi', 'Lucca', 'Macerata', 'Mantova', 'Massa-Carrara', 'Matera', 'Messina', 'Milano', 'Modena', 'Monza-Brianza', 'Napoli', 'Novara', 'Nuoro', 'Oristano', 'Padova', 'Palermo', 'Parma', 'Pavia', 'Perugia', 'Pesaro Urbino', 'Pescara', 'Piacenza', 'Pisa', 'Pistoia', 'Pordenone', 'Potenza', 'Prato', 'Ragusa', 'Ravenna', 'Reggio Calabria', 'Reggio Emilia', 'Rieti', 'Rimini', 'Roma', 'Rovigo', 'Salerno', 'Sassari', 'Savo

In [10]:
print("Rows:", df_provinces.shape[0])
print("Unique provinces:", df_provinces["Province"].nunique())
print("Years:", df_provinces["Year"].unique())


Rows: 1054
Unique provinces: 106
Years: <StringArray>
['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
 '2023']
Length: 10, dtype: str


In [ ]:
duplicates = df_provinces.duplicated(
    subset=["Province", "Year"],
    keep=False
)

print("Duplicate Province-Year pairs:", duplicates.sum()) # Should be 0 if no duplicates


Duplicate Province-Year pairs: 0


In [12]:
df_provinces = df_provinces.sort_values(
    by=["Province", "Year"]
).reset_index(drop=True)

In [ ]:
cols = df_provinces.columns.tolist()
cols.insert(0, cols.pop(cols.index("Province"))) # Move "Province" to the front
df_provinces = df_provinces[cols]

In [ ]:
df_provinces = df_provinces.drop(columns=["Location"])  
df_provinces.columns = [col.replace("Price", "") if "Price" in col else col for col in df_provinces.columns] # Remove "Price" from column names

In [15]:
df_provinces =  df_provinces.groupby("Province", as_index=False).max()
df_provinces = df_provinces.drop(columns=["Year"])

In [16]:
df_provinces.head()

,Province,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Agrigento,5.1,5.0,4.9,5.3,4.8,5.2,5.1,6.1,5.7,6.3
1,Alessandria,5.0,5.2,5.3,5.3,5.3,5.1,5.3,5.4,5.4,5.6
2,Ancona,7.1,6.9,7.1,7.1,6.8,7.2,7.3,7.4,7.9,8.4
3,Aosta,10.5,10.2,8.8,9.8,11.8,10.6,10.7,13.2,15.5,19.1
4,Arezzo,7.0,7.1,6.8,7.1,6.7,7.1,7.2,7.1,7.7,8.5


Check for missing values before exporting the dataset.

In [17]:
df_missing = df_provinces.isnull().sum()
print("Missing values per column:"+str(df_missing))

Missing values per column:Province    0
 2014       1
 2015       0
 2016       1
 2017       0
 2018       0
 2019       0
 2020       0
 2021       2
 2022       1
 2023       1
dtype: int64


In [18]:
provinces_with_missing = df_provinces[df_provinces.isna().any(axis=1)] # Filter provinces with any missing values

print("Number of provinces with missing data:",
      provinces_with_missing.shape[0])
print(provinces_with_missing["Province"].tolist())

missing_detail = provinces_with_missing.set_index("Province")

print(missing_detail[missing_detail.isna()])


Number of provinces with missing data: 3
['Bolzano-Bozen', 'Isernia', 'Oristano']
               2014  2015  2016  2017  2018  2019  2020  2021  2022  2023
Province                                                                 
Bolzano-Bozen   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN
Isernia         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN
Oristano        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN


In [25]:
# Replace missing values with mean of column before and column after
# Ensure year columns are sorted numerically
year_cols = sorted(
    [col for col in df_provinces.columns if col != "Province"],
    key=int
)

df_provinces = df_provinces[["Province"] + year_cols]


In [26]:
df_provinces[year_cols] = (
    df_provinces[year_cols]
    .astype(float)
    .interpolate(
        method="linear",
        axis=1,
        limit_direction="both"
    )
) # Interpolate missing values linearly across years, allowing extrapolation at ends

df_provinces[year_cols] = df_provinces[year_cols].round(2) # Round to 2 decimal places for cleaner presentation

In [27]:
df_missing = df_provinces.isnull().sum()
print("Missing values per column:"+str(df_missing))

Missing values per column:Province    0
 2014       0
 2015       0
 2016       0
 2017       0
 2018       0
 2019       0
 2020       0
 2021       0
 2022       0
 2023       0
dtype: int64


In [ ]:
df_provinces.info() # One last verification

<class 'pandas.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Province  106 non-null    str    
 1    2014     106 non-null    float64
 2    2015     106 non-null    float64
 3    2016     106 non-null    float64
 4    2017     106 non-null    float64
 5    2018     106 non-null    float64
 6    2019     106 non-null    float64
 7    2020     106 non-null    float64
 8    2021     106 non-null    float64
 9    2022     106 non-null    float64
 10   2023     106 non-null    float64
dtypes: float64(10), str(1)
memory usage: 9.2 KB


Data has been cleaned and is ready for exporting.

Note: South Sardinia is the only province missing

In [29]:
output_file = "rent_prices.csv"

df_provinces.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"File saved as: {output_file}")

File saved as: rent_prices.csv
